In [1]:


import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression

MY_FILE = "../air-quality.csv"

raw = pd.read_csv(MY_FILE)

TARGET = "co_gt"
DROP = ["datetime", "nmhc_gt", "c6h6_gt", "nox_gt", "no2_gt", "absolute_humidity"]

df = raw.drop(columns=DROP)

df = df.replace(-200, np.nan)
df = df.dropna(subset=[TARGET])

y = df[TARGET]
X = df.drop(columns=[TARGET])

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=0
)

numeric_cols = X.select_dtypes("number").columns.tolist()
text_cols = [c for c in X.columns if c not in numeric_cols]

preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())
    ]), numeric_cols),

    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), text_cols)
])

pipe = Pipeline([
    ("prep", preprocess),
    ("model", LinearRegression())
])

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=0
)

METRIC = "neg_mean_absolute_error"

In [2]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error, r2_score

# 5-fold cross-validation
cv_scores = cross_val_score(
    pipe, X_train, y_train,
    cv=cv,
    scoring=METRIC
)

print("Cross-validation MAE:", -cv_scores.mean())

# Train the model
pipe.fit(X_train, y_train)

# Predict using test data
y_pred = pipe.predict(X_test)

# Evaluate performance
print("Test MAE:", mean_absolute_error(y_test, y_pred))
print("Test R²:", r2_score(y_test, y_pred))

Cross-validation MAE: 0.3696228343218377
Test MAE: 0.3565090402249915
Test R²: 0.8179460799595999


In [ ]:

from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV

# Create decision tree pipeline
tree_pipe = Pipeline([
    ("prep", preprocess),
    ("model", DecisionTreeRegressor(random_state=0))
])

# Hyperparameters to test
grid = {
    "model__max_depth": range(1,50),
    "model__min_samples_leaf":range(1,50)
}

# 5-fold cross-validation and tuning
search = GridSearchCV(
    tree_pipe,
    grid,
    cv=cv,
    scoring=METRIC,
    return_train_score=True
)

search.fit(X_train, y_train)

# Evaluate cross-validation results
best = search.best_index_

print("Best parameters:", search.best_params_)
print("Training MAE:", -search.cv_results_["mean_train_score"][best])
print("Cross-validation MAE:", -search.best_score_)

# Computational efficiency
print("Training time (s):", search.cv_results_["mean_fit_time"][best])
print("Prediction time (s):", search.cv_results_["mean_score_time"][best])
